# Notebook 01 -- Data cleaning and processing

## Imports

In [57]:
import polars as pl
import seaborn as sns
from pathlib import Path
from matplotlib import pyplot as plt

print(f"Polars version: {pl.__version__}")

Polars version: 1.44.2


## Route vars
Simple file constants to define ahead of time.

In [58]:
# directory paths
PROJECT_ROOT = Path("..").resolve()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Outage constans
OUTAGE_PCT_THRESHOLD = 0.02
OUTAGE_CNT_THRESHOLD = 500

## Scan raw data
Prefer to scan into a lazy dataframe give the size of the data being handled.

In [59]:
raw_files = list(RAW_DATA_DIR.glob("*.csv"))
print(f'{len(raw_files)} raw files found in data/raw')
for f in raw_files:
    print(f" - {f.name} ({f.stat().st_size / (1024 * 1024):.1f} MB)")

if raw_files:
    lazy_df = pl.scan_csv(
        RAW_DATA_DIR / "*.csv", 
        try_parse_dates=True
    )
    for col, dtype in lazy_df.collect_schema().items():
        print(f" {col:20s}: {dtype}")

else:
    print("Raw data could not be located.")

1 raw files found in data/raw
 - eaglei_outages_2025.csv (1337.4 MB)
 fips_code           : Int64
 county              : String
 state               : String
 customers_out       : Int64
 run_start_time      : Datetime(time_unit='us', time_zone=None)


## Column name consts
For convenience.

In [60]:
COL_TIMESTAMP = "run_start_time" 
COL_FIPS = "fips_code" 
COL_CUSTOMERS_OUT = "customers_out" 

## Hourly Downsample
The EAGLI-I dataset uses 15-minute logging increments, which is far too much data to process. Instead, we will truncate it down to 1hr increments to still obtain the information we want while significanly reducing overhead.

In [61]:
hourly_timestamp = pl.col(COL_TIMESTAMP).dt.truncate("1h").alias("hourly_timestamp")

hourly_lazy = (
    lazy_df
    .group_by([pl.col(COL_FIPS), hourly_timestamp])
    .agg([
        pl.col(COL_CUSTOMERS_OUT).max().alias("customers_out_max"),
        pl.col("county").first().alias("county"),
        pl.col("state").first().alias("state"),
        pl.len().alias("intervals_reported")
    ])
    .sort([pl.col(COL_FIPS),
    pl.col("hourly_timestamp")])
)

preview_df = hourly_lazy.collect().head()
print(preview_df)

shape: (5, 6)
┌───────────┬─────────────────────┬───────────────────┬─────────┬─────────┬────────────────────┐
│ fips_code ┆ hourly_timestamp    ┆ customers_out_max ┆ county  ┆ state   ┆ intervals_reported │
│ ---       ┆ ---                 ┆ ---               ┆ ---     ┆ ---     ┆ ---                │
│ i64       ┆ datetime[μs]        ┆ i64               ┆ str     ┆ str     ┆ u32                │
╞═══════════╪═════════════════════╪═══════════════════╪═════════╪═════════╪════════════════════╡
│ 1001      ┆ 2025-01-01 00:00:00 ┆ 7                 ┆ Autauga ┆ Alabama ┆ 4                  │
│ 1001      ┆ 2025-01-01 01:00:00 ┆ 13                ┆ Autauga ┆ Alabama ┆ 4                  │
│ 1001      ┆ 2025-01-01 02:00:00 ┆ 405               ┆ Autauga ┆ Alabama ┆ 4                  │
│ 1001      ┆ 2025-01-01 03:00:00 ┆ 7                 ┆ Autauga ┆ Alabama ┆ 4                  │
│ 1001      ┆ 2025-01-01 04:00:00 ┆ 7                 ┆ Autauga ┆ Alabama ┆ 3                  │
└───────────┴───

## Target column
We can add target column to see which reports meet our criteria of an outage (500 indivuals OR 2% of county population).

In [62]:
hourly_with_target = (
    hourly_lazy
    .with_columns([
        (pl.col("customers_out_max") >= OUTAGE_CNT_THRESHOLD)
        .cast(pl.Int8)
        .alias("is_outage_500")
    ])
)

preview_target_df = hourly_with_target.head(10).collect()
print(preview_target_df)

shape: (10, 7)
┌───────────┬────────────────┬────────────────┬─────────┬─────────┬────────────────┬───────────────┐
│ fips_code ┆ hourly_timesta ┆ customers_out_ ┆ county  ┆ state   ┆ intervals_repo ┆ is_outage_500 │
│ ---       ┆ mp             ┆ max            ┆ ---     ┆ ---     ┆ rted           ┆ ---           │
│ i64       ┆ ---            ┆ ---            ┆ str     ┆ str     ┆ ---            ┆ i8            │
│           ┆ datetime[μs]   ┆ i64            ┆         ┆         ┆ u32            ┆               │
╞═══════════╪════════════════╪════════════════╪═════════╪═════════╪════════════════╪═══════════════╡
│ 1001      ┆ 2025-01-01     ┆ 7              ┆ Autauga ┆ Alabama ┆ 4              ┆ 0             │
│           ┆ 00:00:00       ┆                ┆         ┆         ┆                ┆               │
│ 1001      ┆ 2025-01-01     ┆ 13             ┆ Autauga ┆ Alabama ┆ 4              ┆ 0             │
│           ┆ 01:00:00       ┆                ┆         ┆         ┆         

## Processing with Parquet

In [ ]:
# Output path for the processed hourly dataset
PROCESSED_PARQUET = PROCESSED_DATA_DIR / "eaglei_2025_hourly.parquet"

print(f"Executing pipeline and streaming to Parquet: {PROCESSED_PARQUET}")
print("This processes the full 1.34 GB CSV in streaming batches...")

# Stream directly to Parquet to keep memory usage near zero
hourly_with_target.sink_parquet(
    PROCESSED_PARQUET,
    compression="zstd"
)

# Benchmark the file size reduction
csv_size_mb = (RAW_DATA_DIR / "eaglei_outages_2025.csv").stat().st_size / (1024 * 1024)
parquet_size_mb = PROCESSED_PARQUET.stat().st_size / (1024 * 1024)

print("\nProcessing Complete!")
print(f"  Raw CSV size:        {csv_size_mb:,.1f} MB")
print(f"  Processed Parquet:   {parquet_size_mb:,.1f} MB")
print(f"  Compression factor:  {csv_size_mb / parquet_size_mb:.1f}x reduction")
